In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import display
import ipywidgets as widgets

# Optional: Configure plot resolution
# %matplotlib inline
# plt.rcParams['figure.dpi'] = 100


In [2]:
print(os.getcwd())
os.chdir('/home/ubuntu/work/saved_data/Global-Canopy-Height-Map/')
print(os.getcwd())

/home/ubuntu/work/saved_data/Global-Canopy-Height-Map/prediction
/home/ubuntu/work/saved_data/Global-Canopy-Height-Map


In [3]:
def visualize_prediction_interactive(entry, rgb_channels=[6, 5, 4]):
    file_name = entry["file_name"]
    model_variant = entry["variant"]
    outputs = entry["outputs"]
    x_input = np.array(entry["input_image"])  # Already a NumPy array

    # Normalize input for visualization
    # x_input_normalized = x_input / np.percentile(x_input, 99)
    # x_input_normalized = np.clip(x_input_normalized, 0, 1)
    x_input_normalized = np.clip(x_input / 3000, 0, 1)

    fig, axs = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

    # 🔹 Input RGB Image
    axs[0].imshow(x_input_normalized[rgb_channels, :, :].transpose(1, 2, 0))
    axs[0].set_title(f"Input Image: {file_name}")
    axs[0].axis("off")

    # 🔹 Middle & Right Panels
    if model_variant == "baseline":
        mean_prediction = outputs["mean"]
        im = axs[1].imshow(mean_prediction, cmap="viridis", vmin=0, vmax=35)
        axs[1].set_title("Baseline Prediction")
        axs[2].axis("off")

    elif model_variant == "ensemble":
        # mean_prediction = outputs["mean"]
        # variance = outputs["variance"]

        # im = axs[1].imshow(mean_prediction, cmap="viridis", vmin=0, vmax=35)
        # axs[1].set_title("Ensemble Mean")

        # im_var = axs[2].imshow(variance, cmap="magma", vmin=0, vmax=variance.max())
        # axs[2].set_title("Ensemble Variance")

        # divider2 = make_axes_locatable(axs[2])
        # cax2 = divider2.append_axes("right", size="5%", pad=0.05)
        # fig.colorbar(im_var, cax=cax2, orientation="vertical")

        median_prediction = outputs["q50"]
        q10 = outputs["q10"]
        q90 = outputs["q90"]

        im = axs[1].imshow(median_prediction, cmap="viridis", vmin=0, vmax=35)
        axs[1].set_title("Median Prediction (Quantiles)")

        interval_width = np.array(q90) - np.array(q10)
        im_ci = axs[2].imshow(interval_width, cmap="magma", vmin=0, vmax=interval_width.max())
        axs[2].set_title("80% Interval Width (Q90 - Q10)")

        divider2 = make_axes_locatable(axs[2])
        cax2 = divider2.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(im_ci, cax=cax2, orientation="vertical")

    elif model_variant == "quantiles":
        median_prediction = outputs["q50"]
        q10 = outputs["q10"]
        q90 = outputs["q90"]

        im = axs[1].imshow(median_prediction, cmap="viridis", vmin=0, vmax=35)
        axs[1].set_title("Median Prediction (Quantiles)")

        interval_width = np.array(q90) - np.array(q10)
        im_ci = axs[2].imshow(interval_width, cmap="magma", vmin=0, vmax=interval_width.max())
        axs[2].set_title("80% Interval Width (Q90 - Q10)")

        divider2 = make_axes_locatable(axs[2])
        cax2 = divider2.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(im_ci, cax=cax2, orientation="vertical")

    elif model_variant == "gaussian":
        mean_prediction = outputs["mean"]
        lower_bound = outputs["lower_80"]
        upper_bound = outputs["upper_80"]

        im = axs[1].imshow(mean_prediction, cmap="viridis", vmin=0, vmax=35)
        axs[1].set_title("Gaussian Mean")

        interval_width = np.array(upper_bound) - np.array(lower_bound)
        im_ci = axs[2].imshow(interval_width, cmap="magma", vmin=0, vmax=interval_width.max())
        axs[2].set_title("80% Confidence Interval Width")

        divider2 = make_axes_locatable(axs[2])
        cax2 = divider2.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(im_ci, cax=cax2, orientation="vertical")

    else:
        axs[1].set_title(f"Unknown variant: {model_variant}")
        axs[2].axis("off")

    # Add colorbar to prediction
    divider = make_axes_locatable(axs[1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(im, cax=cax, orientation="vertical")

    plt.show()


In [4]:
def stream_predictions_from_folder(folder_path):
    files = sorted(f for f in os.listdir(folder_path) if f.endswith(".npz"))

    def load_entry(idx):
        path = os.path.join(folder_path, files[idx])
        data = np.load(path, allow_pickle=True)
        entry = {k: data[k].item() if data[k].shape == () else data[k] for k in data}
        return entry

    def show(idx):
        entry = load_entry(idx)
        visualize_prediction_interactive(entry)

    return widgets.interact(show, idx=widgets.IntSlider(min=0, max=len(files) - 1, step=1, description="Image"))



In [ ]:
# Example usage:
# Update the path below to your actual prediction folder
stream_predictions_from_folder("prediction_outputs/gaussian")

interactive(children=(IntSlider(value=0, description='Image', max=1175), Output()), _dom_classes=('widget-inte…

<function __main__.stream_predictions_from_folder.<locals>.show(idx)>